# Q-Trust — Rigorous Real-Data Training Campaign

> Reproduces, cell by cell, the flagship numbers in the README / Truth Audit:
> CodeBERTa code discovery on **13,973 real code files**, the **40-fold
> host-disjoint LOO** GNN planner benchmark on 280 real TLS hosts (τ-b 0.7263,
> deterministic kernels, folds sharded across 4 A100s), the PPO migration
> agent on **real-CBOM estates with scan-derived risk labels** (140.34 vs
> doctrine heuristic 140.62 — a tie, Δ −0.28 — vs random 136.84, +2.6%;
> bit-reproducible across processes), the side-channel detector on **real
> liboqs timing traces**, and all 15
> `qtrust_ai` intelligence-layer models trained on real datasets (code corpus
> / TLS scan / NVD vendor data).
>
> **Hardware:** 8× NVIDIA A100-SXM4-80GB · 24 cores · 1.7 TiB RAM (BrevLab)
>
> Run this notebook (or execute it headlessly with `nbconvert --execute`).
> `FULL = False` runs a CI-sized smoke pass; set `FULL = True` for the complete
> campaign. Background jobs launched from cells are tracked with their logs in
> `logs/`; Long-running stages print `tail` of their log so you can watch.

In [1]:
import os

# 0. Configuration
FULL = False            # True = rigorous campaign (LOO 40-fold merge, RL retrain, 4-epoch CodeBERTa)
GPU = "2"               # CUDA_VISIBLE_DEVICES for GPU stages ("" = let torch decide)
N_LOO_EPOCHS = 30 if FULL else 2
N_LOO_SYNTH = 2000 if FULL else 300
N_HF = 4 if FULL else 0   # smoke skips the transformer fine-tune (code detector stays deterministic-layer in smoke)
# 4000 PPO episodes costs ~3 h on one A100 (~2.9 s/rollout); the canonical
# committed agent (rl_agent_real.pt, 2026-09-02) was trained for 190 episodes
# (reward converged 8.86 @100 -> 8.82 @190, best 8.86) on real-CBOM packs with
# scan-derived risk labels and a per-process-deterministic packing. Set 4000
# for a longer-horizon agent.
N_RL_EPISODES = 190 if FULL else 64
N_SIDE_EPOCHS = 60 if FULL else 12
N_ANOMALY_EPOCHS = 120 if FULL else 25
# Smoke mode writes to scratch artifacts so the canonical benchmark JSONs
# (cited by the README / Truth Audit) are only refreshed by FULL runs.
SUFFIX = "" if FULL else "_notebook"
TR = f"qtrust_ai/artifacts/training_report_real{SUFFIX}.json"
BC = f"qtrust_ai/artifacts/benchmark_comparison{SUFFIX}.json"
LOO = f"planner/results/real_cbom_loo{SUFFIX}.json"
RL = f"planner/results/rl_benchmark_real_cbom{SUFFIX}.json"
if GPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU
os.environ.setdefault("QTRUST_DISABLE_COMPILE", "1")  # dynamic graphs: compile is 10x slower
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
print("FULL =", FULL, "| CUDA_VISIBLE_DEVICES =", GPU, "| artifact suffix:", repr(SUFFIX))

FULL = False | CUDA_VISIBLE_DEVICES = 2 | artifact suffix: '_notebook'


## 1. Environment & dataset inventory

In [2]:
import glob
import json
import subprocess
import sys
import time
from pathlib import Path

import torch
import torch_geometric
import transformers

# The notebook may be executed from anywhere (BrevLab root, research/notebooks,
# or a subprocess cwd) — always anchor at the repo root, like the scripts do.
ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    for cand in [ROOT.parent, ROOT.parent.parent]:
        if (cand / "scripts").exists():
            ROOT = cand
            break
if not (ROOT / "scripts").exists():
    raise SystemExit(f"repo root not found from {Path.cwd()}")
print("repo root:", ROOT)
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available(),
      "| devices:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name} ({p.total_memory/1e9:.0f} GiB)")
print("pyg", torch_geometric.__version__, "| transformers", transformers.__version__)
datasets = ROOT / "qtrust_ai" / "artifacts" / "real_datasets"
code_corpus = json.loads((datasets / "code_corpus.json").read_text())
tls = json.loads((datasets / "tls_inventory.json").read_text())
n_crypto = sum(1 for r in code_corpus["corpus"] if r["is_crypto"])
print(f"real code corpus: {len(code_corpus['corpus'])} files ({n_crypto} crypto) "
      f"| languages: {sorted({r['language'] for r in code_corpus['corpus']})}")
print(f"real TLS inventory: {len(tls.get('cboms', []))} CBOMs / {tls.get('n_findings')} findings")
cboms = sorted((ROOT / "planner" / "data" / "real_cboms").glob("*.json"))
print(f"host-disjoint real CBOM corpus: {len(cboms)} CBOMs "
      f"({sum(len(json.loads(open(c).read())['assets']) for c in cboms)} assets)")
traces = sorted(glob.glob("/tmp/real_data/traces_*.txt"))
print(f"real liboqs timing traces: {[Path(t).stem.removeprefix('traces_') for t in traces]}")

/lp-dev/24BCE1793/fignn_env/lib/python3.13/site-packages/torch/cuda/__init__.py:64: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


/lp-dev/24BCE1793/fignn_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


repo root: /home/nvidia/Project
torch 2.13.0+cu130 | cuda available: True | devices: 1
  GPU 0: NVIDIA A100-SXM4-80GB (85 GiB)
pyg 2.8.0 | transformers 5.15.1


real code corpus: 13973 files (7632 crypto) | languages: ['c', 'cpp', 'csharp', 'go', 'java', 'javascript', 'php', 'python', 'rust', 'shell', 'solidity', 'swift', 'typescript']
real TLS inventory: 277 CBOMs / 277 findings
host-disjoint real CBOM corpus: 40 CBOMs (280 assets)
real liboqs timing traces: ['MLDSA44_sign', 'MLDSA44_verify', 'MLKEM512_decaps', 'MLKEM512_encaps', 'MLKEM768_decaps']


## 2. Train all 15 `qtrust_ai` models on the real datasets

Runs `scripts/train_qtrust_all.py --real`: discovery (CodeBERTa fine-tune, GPU), purpose classifier, blast radius, temporal GNN, quantum-exposure risk, PQC recommender, cost/failure/interop (labels proprietary → synthetic, stated), multi-objective RL, anomaly, regression, vendor supply-chain, copilot, policy — **plus** the baseline comparison (models vs naive baselines on the same real splits).

In [3]:
print(">>> train_qtrust_all.py --real (hf_epochs =", N_HF, ")")
t0 = time.time()
r = subprocess.run(
    [sys.executable, "scripts/train_qtrust_all.py", "--real", "--epochs", "5",
     "--hf-epochs", str(N_HF), "--report", TR, "--benchmark-out", BC],
    capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-3000:])
print(r.stderr[-800:] if r.returncode else "", flush=True)
print("exit:", r.returncode, "| wall:", round(time.time()-t0, 1), "s")
rep = json.loads((ROOT / TR).read_text())
s = rep["summary"]
print(f"summary: {s['trained']} trained / {s['anchor_fail']} anchor-fail / {s['errors']} errors")
for e in rep["results"]:
    ev = (e.get("train") or {}).get("_eval") or {}
    if isinstance(ev, dict) and ("accuracy" in ev or "f1" in ev):
        print(f"  {e['model']:<40} { {k: round(ev[k],3) for k in ('accuracy','f1','precision','recall') if k in ev} }")

>>> train_qtrust_all.py --real (hf_epochs = 0 )



=== Loading real datasets ===

  real code corpus: 13973 files (crypto=7632)
  real TLS: 277 findings across 277 hosts
  real vendor: 16 libraries
=== Training all qtrust_ai models (epochs=5, rl_episodes=40, real=True) ===

  [trained     ] discovery/CryptoCodeDetector         28.3s  examples=11558.0 has_torch  anchors 1/1
  [trained     ] discovery/AlgorithmPurposeClassifier    5.3s  examples=935.0 has_sklearn  anchors 1/1
  [trained     ] graph/BlastRadius (calibrate)         0.0s    anchors 1/1
  [trained     ] graph/TemporalGNN                    13.3s  mae=4.018 examples=301.0 has_torch  anchors 1/1
  [trained     ] risk/QuantumExposureModel             0.3s  mae=0.0 examples=277.0 has_sklearn  anchors 1/1
  [trained     ] migration/PQCRecommender              3.9s  examples=935.0 has_sklearn  anchors 1/1
  [trained     ] migration/MigrationCostPredictor      1.2s  mae=16.094 examples=401.0 has_sklearn  anchors 1/1
  [trained     ] migration/MigrationFailurePredictor    0.5s  exa

exit: 0 | wall: 141.6 s
summary: 15 trained / 0 anchor-fail / 0 errors
  discovery/CryptoCodeDetector             {'accuracy': 0.649, 'f1': 0.71, 'precision': 0.971, 'recall': 0.56}
  discovery/AlgorithmPurposeClassifier     {'accuracy': 1.0}
  migration/PQCRecommender                 {'accuracy': 1.0}
  migration/MigrationFailurePredictor      {'accuracy': 0.79}
  monitoring/CryptoAnomalyDetector         {'f1': 0.667, 'precision': 0.5, 'recall': 1.0}
  monitoring/CryptoRegressionDetector      {'accuracy': 1.0, 'f1': 1.0, 'precision': 1.0, 'recall': 1.0}
  copilot/SecurityCopilot                  {'accuracy': 1.0}
  policy/PolicyEngine                      {'accuracy': 1.0}


## 3. GNN planner — honest out-of-sample benchmark (40-fold LOO)

`scripts/eval_real_cbom_loo.py` runs the host-disjoint leave-one-out protocol: for each real CBOM, fine-tune a fresh model on the other 39 (+ synthetic doctrine mix) and evaluate on the held-out estate only. Baselines (priority heuristic, random) are scored on the identical folds. In `FULL` mode the cell first tries to **merge the 4 GPU shards** already produced by the 4-A100 campaign (`--merge-shards`); if they are missing it prints the exact shard commands to launch. Smoke mode runs `--quick` (3 folds) into a suffixed artifact.

In [4]:
print(">>> eval_real_cbom_loo.py (FULL: merge 4-GPU shards; else quick smoke)")
t0 = time.time()
shards = sorted((ROOT / "planner" / "results").glob(f"{Path(LOO).name}_shard*.json"))
if FULL and len(shards) == 4:
    print(f"found {len(shards)} shards -> merging")
    r = subprocess.run([sys.executable, "scripts/eval_real_cbom_loo.py", "--merge-shards", "--out", LOO],
                       capture_output=True, text=True, cwd=ROOT)
    print(r.stdout[-1500:])
    print(r.stderr[-500:] if r.returncode else "")
    print("exit:", r.returncode)
elif FULL:
    print("No 4-GPU shards found. Launch on 4 A100s and merge here, e.g.:")
    print("  CUDA_VISIBLE_DEVICES=0 python scripts/eval_real_cbom_loo.py --epochs 30 --n-synthetic 2000 --fold-start 0  --fold-end 10")
    print("  CUDA_VISIBLE_DEVICES=1 python scripts/eval_real_cbom_loo.py --epochs 30 --n-synthetic 2000 --fold-start 10 --fold-end 20")
    print("  CUDA_VISIBLE_DEVICES=2 python scripts/eval_real_cbom_loo.py --epochs 30 --n-synthetic 2000 --fold-start 20 --fold-end 30")
    print("  CUDA_VISIBLE_DEVICES=3 python scripts/eval_real_cbom_loo.py --epochs 30 --n-synthetic 2000 --fold-start 30 --fold-end 40")
    print("  python scripts/eval_real_cbom_loo.py --merge-shards")
    print("Skipping the single-process 40-fold run here (~3 h); see docs/DEVELOPER_ROADMAP.md.")
else:
    cmd = [sys.executable, "scripts/eval_real_cbom_loo.py", "--quick",
           "--epochs", str(N_LOO_EPOCHS), "--n-synthetic", str(N_LOO_SYNTH), "--out", LOO]
    r = subprocess.run(cmd, capture_output=True, text=True, cwd=ROOT)
    print(r.stdout[-1800:])
    print(r.stderr[-500:] if r.returncode else "", flush=True)
    print("exit:", r.returncode, "| wall:", round(time.time()-t0, 1), "s")
loo_path = ROOT / LOO
if loo_path.exists():
    rep = json.loads(loo_path.read_text())
    a = rep["aggregate"]
    print("aggregate τ-b  model", round(a["model"]["kendall"]["mean"],4),
          "| heuristic", round(a["heuristic"]["kendall"]["mean"],4),
          "| random", round(a["random"]["kendall"]["mean"],4),
          "| Δmodel-heur", round(rep["model_vs_heuristic_tau_b"],4))

>>> eval_real_cbom_loo.py (FULL: merge 4-GPU shards; else quick smoke)


d from checkpoint: /home/nvidia/Project/planner/model_gpu_v3.pt (prior lineage: 208 epochs, 200,000 graphs)
Model parameters: 237,070 (norm=layer)
Epoch   1/2 | loss=1.9755 lr=5.00e-04 | val tau=0.9145 top5=0.913 top10=0.877 | GPU=0.0GB | elapsed=1s | data_hash=b6b582666dbe0378
  -> Saved best model (tau=0.9145)
Epoch   2/2 | loss=1.5313 lr=5.00e-05 | val tau=0.9369 top5=0.907 top10=0.920 | GPU=0.0GB | elapsed=1s | data_hash=b6b582666dbe0378
  -> Saved best model (tau=0.9369)

Training complete in 1s (0.0 min)
Best validation Kendall tau: 0.9369
Model saved to: /home/nvidia/Project/planner/_loo_fold1.pt (data_hash=b6b582666dbe0378)
  fold  2/3 real_finance_1.json: model τ-b=0.5000 | heuristic=0.5000 | random=0.2143 | n=8 | 5s elapsed
Device: NVIDIA A100-SXM4-80GB
GPU memory: 85.1 GB
Generating 300 synthetic graphs...
Added 2 real-data graphs (total 302)
Generation took 0.5s
Train: 272 graphs, Val: 30 graphs
Initialized from checkpoint: /home/nvidia/Project/planner/model_gpu_v3.pt (prio

exit: 0 | wall: 15.2 s
aggregate τ-b  model 0.6546 | heuristic 0.6546 | random 0.1365 | Δmodel-heur 0.0


## 4. RL migration agent — retrain on real CBOMs, benchmark vs heuristic/random

Real TLS findings carry only the CBOM builder's blanket `criticality: medium`, which would leave the reward with no sequencing signal — the retrain and eval scripts derive risk labels from the real scan fields (`risk_criticality_from_scan`: RSA-1024 → critical, RSA-2048 → high, expired/self-signed/near-expiry raise the class). `scripts/retrain_rl_real_cbom.py` runs PPO (64 vectorized envs, deterministic kernels) on 100 packed real-CBOM estates; `scripts/eval_rl_real_cbom.py` then does greedy rollouts of agent vs criticality heuristic vs random on 40 packed real environments.

In [5]:
if FULL:
    print(">>> retrain_rl_real_cbom.py", N_RL_EPISODES, "episodes")
    r = subprocess.run([sys.executable, "scripts/retrain_rl_real_cbom.py", str(N_RL_EPISODES)],
                      capture_output=True, text=True, cwd=ROOT)
    print(r.stdout[-1500:])
    print(r.stderr[-600:] if r.returncode else "")
    print("exit:", r.returncode)
print(">>> eval_rl_real_cbom.py")
r = subprocess.run([sys.executable, "scripts/eval_rl_real_cbom.py", "--out", RL],
                   capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-2500:])
print(r.stderr[-600:] if r.returncode else "", flush=True)
print("exit:", r.returncode)

>>> eval_rl_real_cbom.py


real-asset criticality (scan-derived): {'critical': 77, 'high': 179, 'medium': 23}
Device: cuda · 40 packed real-CBOM environments (pack_seed=99)

  env    n    agent heuristic   random  delta_vs_heur
    0   18   144.08    144.08   144.18  +0.00
    1   15   142.01    142.01   142.01  +0.00
    2   13   124.03    124.03   124.39  +0.00
    3   15   145.34    145.34   145.40  +0.00
    4   20   137.60    137.80   132.88  -0.20
    5   24   136.26    136.26   121.71  +0.00
    6   18   137.14    137.14   127.67  +0.00
    7   11   143.13    143.13   143.56  +0.00
    8   22   133.65    133.65   128.80  +0.00
    9   26   138.61    138.51   128.61  +0.10
   10   23   165.53    165.53   156.10  +0.00
   11   11   144.42    144.42   144.63  +0.00
   12    8   138.26    138.36   133.59  -0.09
   13   23   138.62    138.62   133.80  +0.00
   14   14   134.64    134.79   129.93  -0.16
   15   21   158.87    158.87   153.99  +0.00
   16    8   139.02    139.02   139.37  +0.00
   17    6   134.

exit: 0


## 5. Side-channel detector on real liboqs traces

Trains a CNN on bootstrap-resampled windows of **real** liboqs ML-KEM-512/768 and ML-DSA-44 timing traces (+ keyed leak injection on the same noise floor), then validates clean vs leak-injected held-out real trace sets.

In [6]:
print(">>> train_real_side_channel.py", N_SIDE_EPOCHS, "epochs")
SIDE_SAVE = "inspector/side_channel_model_real.pt" if FULL else             "inspector/side_channel_model_real_notebook.pt"  # smoke never clobbers canonical
r = subprocess.run([sys.executable, "scripts/train_real_side_channel.py",
                    "--traces-dir", "/tmp/real_data", "--epochs", str(N_SIDE_EPOCHS),
                    "--save-path", SIDE_SAVE],
                   capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-2200:])
print(r.stderr[-600:] if r.returncode else "", flush=True)
print("exit:", r.returncode, "| saved:", SIDE_SAVE)

>>> train_real_side_channel.py 12 epochs


Loaded 5 real trace sets: ['MLDSA44_sign', 'MLDSA44_verify', 'MLKEM512_decaps', 'MLKEM512_encaps', 'MLKEM768_decaps']
  MLDSA44_sign: n=10000 mean=101.4us std=62.0us
  MLDSA44_verify: n=10000 mean=36.3us std=5.1us
  MLKEM512_decaps: n=10000 mean=16.4us std=2.7us
  MLKEM512_encaps: n=10000 mean=13.3us std=2.9us
  MLKEM768_decaps: n=10000 mean=27.1us std=3.4us
Building features: 2000 clean + 2000 leaking...
Epoch 10/12: loss=0.0203
Epoch 12/12: loss=0.0851
Model saved to inspector/side_channel_model_real_notebook.pt
Calibration anchors: clean=0.001 leak=0.998

Validation on real implementations:
      MLDSA44_sign: leakage_prob=0.0574 -> VERIFIED
    MLDSA44_verify: leakage_prob=0.0502 -> VERIFIED
   MLKEM512_decaps: leakage_prob=0.0541 -> VERIFIED
   MLKEM512_encaps: leakage_prob=0.0643 -> VERIFIED
   MLKEM768_decaps: leakage_prob=0.0499 -> VERIFIED

Validation on leak-injected real traces (amp=0.5):
      MLDSA44_sign: leakage_prob=0.8999 -> HIGH_RISK
    MLDSA44_verify: leakage_prob=0

exit: 0 | saved: inspector/side_channel_model_real_notebook.pt


## 6. Anomaly detector on the real host-disjoint CBOM corpus

VAE trained on 80% of the real CBOMs (per-host), evaluated on the held-out 20% plus three real-world attack injections (weak-key rollback, config drift, renewal failure).

In [7]:
print(">>> train_real_models.py --model anomaly (epochs =", N_ANOMALY_EPOCHS, ")")
r = subprocess.run([sys.executable, "scripts/train_real_models.py", "--model", "anomaly",
                    "--epochs", str(N_ANOMALY_EPOCHS)], capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-2000:])
print(r.stderr[-600:] if r.returncode else "", flush=True)
print("exit:", r.returncode)
if not FULL:
    print("NOTE: smoke retrains the canonical anomaly_model_real.pt at reduced epochs;",
          "re-run `python scripts/train_real_models.py --model anomaly --epochs 120` for the canonical artifact")

>>> train_real_models.py --model anomaly (epochs = 25 )


Loaded 280 real assets -> 279 normalized assets

=== Anomaly detector ===
Epoch 20/25: loss=120.24
Anomaly threshold set to 0.0671 (95th pct of per-CBOM maxima)
Model saved to /home/nvidia/Project/inspector/anomaly_model_real.pt

Held-out normal CBOMs (expected: not anomalous):
  false positives: 4/54

Injected-anomaly CBOMs (expected: flagged):
  pattern 1: score=1.0000 flagged=True
  pattern 1: score=1.0000 flagged=True
  pattern 1: score=1.0000 flagged=True
  pattern 1: score=1.0000 flagged=True
  pattern 1: score=1.0000 flagged=True
  pattern 1: score=1.0000 flagged=True
  pattern 1: score=1.0000 flagged=True
  pattern 1: score=1.0000 flagged=True
  pattern 1: score=1.0000 flagged=True

Detection rate: 162/162 (100%), FPR 4/54




exit: 0
NOTE: smoke retrains the canonical anomaly_model_real.pt at reduced epochs; re-run `python scripts/train_real_models.py --model anomaly --epochs 120` for the canonical artifact


## 7. Recap — the honest headline table

Every number above is out-of-sample on real data (host-disjoint CBOMs, repo-disjoint code splits, held-out liboqs trace sets), with device + config recorded in the artifacts. The GNN reproduces the migration doctrine on 38/40 held-out real CBOMs (τ-b 0.7263) and the RL agent matches the doctrine heuristic on real estates (Δ −0.28) while beating random (+2.6%) — neither beats the doctrine, because the doctrine is the label; the ceiling break requires expert pairwise labels (`QTrust-RiskBench`).

In [8]:
print("campaign artifacts:")
for name in [TR, BC, LOO, RL]:
    p = ROOT / name
    if p.exists():
        d = json.loads(p.read_text())
        when = d.get("generated_at", "?")
        dev = d.get("device", "?")
        print(f"  {name:<58} {when}  [{dev}]")
print("\nNow audit against docs/TRUTH_AUDIT.md — and enjoy the GPUs. 🚀")

campaign artifacts:
  qtrust_ai/artifacts/training_report_real_notebook.json     2026-09-02T08:22:33.563343+00:00  [?]
  qtrust_ai/artifacts/benchmark_comparison_notebook.json     ?  [?]
  planner/results/real_cbom_loo_notebook.json                2026-09-02T08:23:58Z  [NVIDIA A100-SXM4-80GB]
  planner/results/rl_benchmark_real_cbom_notebook.json       2026-09-02T08:24:06Z  [cuda]

Now audit against docs/TRUTH_AUDIT.md — and enjoy the GPUs. 🚀
